
# Tiny Decoder-Only GPT from Scratch (char-level) — with a Hand-Rolled KV-Cache

This notebook builds a **GPT-style decoder-only transformer** from scratch in PyTorch,
trains it as a **character-level language model**, and implements **KV-caching**
ourselves (no `model.generate()` shortcuts from a library) so we can see exactly
why and how it speeds up autoregressive generation.

### What you'll find here
1. **Data**: character-level tokenizer over the Tiny Shakespeare dataset
2. **Model**: multi-head causal self-attention, MLP, residual blocks — built from `nn.Linear`/`nn.LayerNorm` primitives (no `nn.MultiheadAttention`, no `torch.nn.Transformer`)
3. **Training loop**: AdamW, cosine LR schedule, gradient clipping, train/val split, loss curves
4. **KV-cache**: implemented inside the attention module itself, toggled by a `kv_cache` argument
5. **Correctness check**: we *prove* the cached path produces identical logits to the uncached path (this is the part most from-scratch tutorials skip, and the part most likely to be silently wrong)
6. **Benchmark**: cached vs. uncached generation speed, and why the gap grows with sequence length
7. **Sampling**: temperature + top-k text generation from the trained model

### Model scale
With the default config (6 layers, 6 heads, 384-dim embeddings, context length 256) this
model has **~10.8M parameters** — comfortably inside the 10–30M target. A commented-out
"bigger" config gets you to ~20M+ if you have a GPU and want better samples.

### Why char-level?
Character-level modeling means no separate tokenizer/BPE step — the vocabulary is just
the unique characters in the text (~65 for Shakespeare). This keeps the focus on the
*architecture and KV-cache mechanics* rather than tokenization. The same `TinyGPT` class
would work unchanged with a subword vocabulary.



## 1. Setup

We need PyTorch. We'll auto-detect a GPU (`cuda`) or Apple Silicon (`mps`) and fall back
to CPU. Training a 10M-parameter char model on CPU is slow but workable for a few thousand
steps; on a GPU, training to decent samples takes a couple of minutes.


In [ ]:

import math
import os
import time
import urllib.request

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)



## 2. Data: Tiny Shakespeare, character-level

We download the classic ["Tiny Shakespeare"](https://github.com/karpathy/char-rnn) dataset
(~1.1MB of text) and build a character-level vocabulary: every unique character becomes a
token. This gives a small, clean vocabulary (~65 symbols) and lets the model learn spelling,
punctuation, and dialogue structure from raw characters.


In [ ]:

data_path = "input.txt"
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

if not os.path.exists(data_path):
    urllib.request.urlretrieve(url, data_path)

with open(data_path, "r", encoding="utf-8") as f:
    text = f.read()

print(f"Dataset length (characters): {len(text):,}")
print(text[:300])


In [ ]:

# Build the character-level vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocab size: {vocab_size}")
print("".join(chars))

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

# sanity check
assert decode(encode("Hello, world!")) == "Hello, world!"

data = torch.tensor(encode(text), dtype=torch.long)
print("Encoded data shape:", data.shape, data.dtype)


In [ ]:

# Train / validation split (90 / 10)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"train tokens: {len(train_data):,}  val tokens: {len(val_data):,}")


In [ ]:

def get_batch(split, block_size, batch_size, device):
    """Sample a random batch of (input, target) sequences.

    For each sequence we take `block_size` consecutive characters as input and
    the same window shifted one character to the right as the target — i.e.
    at every position the model must predict the *next* character.
    """
    d = train_data if split == "train" else val_data
    ix = torch.randint(0, len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i : i + block_size] for i in ix])
    y = torch.stack([d[i + 1 : i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

# quick smoke test
xb, yb = get_batch("train", block_size=8, batch_size=4, device=device)
print("inputs:\n", xb)
print("targets:\n", yb)
print("example decoded input :", repr(decode(xb[0].tolist())))
print("example decoded target:", repr(decode(yb[0].tolist())))



## 3. The Model

We build the model bottom-up:

- **`CausalSelfAttention`** — multi-head self-attention, masked so position *i* can only
  attend to positions `<= i`. This is where the KV-cache logic lives (explained in detail
  below).
- **`MLP`** — the position-wise feed-forward network (`Linear -> GELU -> Linear`), expanding
  to 4x the embedding dimension, standard in GPT-style transformers.
- **`Block`** — one transformer block: `x = x + Attn(LN(x))`, then `x = x + MLP(LN(x))`.
  This is **pre-norm** (LayerNorm before the sub-layer, not after), which is what GPT-2 and
  essentially all modern decoder-only transformers use — it trains much more stably than
  the original "post-norm" Transformer.
- **`TinyGPT`** — token embedding + learned positional embedding, a stack of `Block`s, a
  final LayerNorm, and a linear "head" projecting back to vocabulary logits.

### How causal masking works (no cache)
For a sequence of length `T`, we compute attention scores for **every** query against
**every** key (`T x T` matrix), then zero out (set to `-inf` before softmax) any entry
where the key position is *after* the query position. This is what makes it "causal":
token `i` can only look at tokens `0..i`, never the future. This is the standard training
path — we feed in a whole chunk of text at once and let the model predict the next
character at *every* position in parallel.


In [ ]:

class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention with optional KV-cache support.

    During training / full-sequence forward passes, `kv_cache` is None and we
    run standard causal attention over the whole sequence (the "no-cache" path).

    During incremental decoding, the caller passes a `kv_cache` dict for this
    layer. We compute K, V only for the *new* token(s), concatenate them onto
    whatever is already stored in the cache, attend the new query/queries
    against the *full* (cached + new) key/value set, and write the updated
    K, V back into the cache for next time.
    """

    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        self.n_head = n_head
        self.n_embd = n_embd
        self.head_dim = n_embd // n_head
        self.block_size = block_size

        # one fused linear layer producing q, k, v all at once (faster than 3 separate)
        self.qkv_proj = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.out_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # lower-triangular causal mask, precomputed once and reused (no-cache path only)
        mask = torch.tril(torch.ones(block_size, block_size, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x, kv_cache=None):
        B, T, C = x.shape  # batch, sequence length (of THIS call), embedding dim
        qkv = self.qkv_proj(x)                       # (B, T, 3C)
        q, k, v = qkv.split(self.n_embd, dim=2)       # each (B, T, C)

        # split into heads: (B, T, C) -> (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        if kv_cache is not None:
            # ============== KV-CACHE PATH ==============
            # How many tokens were already cached before this call?
            T_past = 0 if kv_cache.get("k") is None else kv_cache["k"].size(2)

            # Append this call's new K, V onto whatever's already cached.
            if kv_cache.get("k") is not None:
                k = torch.cat([kv_cache["k"], k], dim=2)  # (B, nh, T_past+T, hd)
                v = torch.cat([kv_cache["v"], v], dim=2)
            kv_cache["k"] = k
            kv_cache["v"] = v

            T_q = q.size(2)   # number of NEW query positions in this call (often 1)
            T_k = k.size(2)   # total cached length so far = T_past + T_q

            att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, nh, T_q, T_k)

            # We still need a causal mask whenever T_q > 1 (e.g. during "prefill",
            # when we feed a multi-token prompt through the cache for the first
            # time): new query i must not see new key j > i. When T_q == 1 (the
            # common single-token decode step) every key is already in the past,
            # so no masking is needed at all.
            if T_q > 1:
                q_pos = torch.arange(T_past, T_past + T_q, device=x.device).unsqueeze(1)
                k_pos = torch.arange(0, T_k, device=x.device).unsqueeze(0)
                causal = k_pos <= q_pos  # (T_q, T_k) bool, True = allowed to attend
                att = att.masked_fill(~causal, float("-inf"))

            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            out = att @ v  # (B, nh, T_q, hd)

        else:
            # ============== NO-CACHE / TRAINING PATH ==============
            att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, nh, T, T)
            causal = self.causal_mask[:T, :T]
            att = att.masked_fill(~causal, float("-inf"))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            out = att @ v  # (B, nh, T, hd)

        # merge heads back: (B, n_head, T_q, head_dim) -> (B, T_q, C)
        out = out.transpose(1, 2).contiguous().view(B, -1, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


In [ ]:

class MLP(nn.Module):
    """Position-wise feed-forward network: Linear -> GELU -> Linear.

    Standard GPT-style design: expand to 4x the embedding dim, apply a
    nonlinearity, project back down. Applied independently at every position
    (it has no notion of sequence order — all the "mixing across positions"
    happens in attention).
    """

    def __init__(self, n_embd, dropout=0.0):
        super().__init__()
        self.fc1 = nn.Linear(n_embd, 4 * n_embd)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(self.act(self.fc1(x))))


class Block(nn.Module):
    """One transformer block, pre-norm style:
        x = x + Attention(LayerNorm(x))
        x = x + MLP(LayerNorm(x))
    Pre-norm (LN before the sublayer) trains much more stably than the
    original post-norm Transformer and is what GPT-2/3 and most modern
    decoder-only models use.
    """

    def __init__(self, n_embd, n_head, block_size, dropout=0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x, kv_cache=None):
        x = x + self.attn(self.ln1(x), kv_cache=kv_cache)
        x = x + self.mlp(self.ln2(x))
        return x



### The full model: `TinyGPT`

Putting the blocks together:

- **Token embedding**: maps each character id to a learned vector.
- **Positional embedding**: a learned vector per absolute position (0, 1, 2, ...) added to
  the token embedding, so the model knows *where* in the sequence each token sits (without
  this, attention would be permutation-invariant and could not tell "AB" from "BA").
- A stack of `n_layer` `Block`s.
- A final `LayerNorm` and a linear `head` that projects back to vocabulary-sized logits.

Note the **`start_pos`** argument to `forward`: when we're not caching, position 0 is
always the start of whatever chunk we feed in. But when we're decoding incrementally with
a cache, the single new token we feed in at step *t* needs positional embedding `t`, not
`0` — `start_pos` tells the model "this chunk begins at absolute position `start_pos` in
the full sequence so far."


In [ ]:

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer=6, n_head=6, n_embd=384, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.n_layer = n_layer

        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def forward(self, idx, targets=None, kv_caches=None, start_pos=0):
        """
        idx:        (B, T) token ids for THIS forward call only.
                    - training / prefill: T can be up to block_size
                    - incremental decode step: T is usually 1
        kv_caches:  list of per-layer dicts (length n_layer), or None.
                    Pass the SAME list object across calls so the cache accumulates.
        start_pos:  absolute position of idx[:, 0] in the full sequence generated
                    so far. Needed so positional embeddings stay correct when
                    decoding token-by-token.
        """
        B, T = idx.shape
        positions = torch.arange(start_pos, start_pos + T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(positions)[None, :, :]
        x = self.drop(x)

        for i, block in enumerate(self.blocks):
            layer_cache = kv_caches[i] if kv_caches is not None else None
            x = block(x, kv_cache=layer_cache)

        x = self.ln_f(x)
        logits = self.head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    def new_kv_cache(self):
        """Create an empty cache: one {k: None, v: None} dict per layer."""
        return [dict(k=None, v=None) for _ in range(self.n_layer)]



## 4. Instantiate the model and check the parameter count

The default config below (`n_layer=6, n_head=6, n_embd=384, block_size=256`) lands at
roughly **10.8M parameters** — inside the requested 10–30M range and quick enough to train
a few thousand steps even on CPU.

If you have a GPU, try the commented-out **bigger config** (`n_layer=8, n_head=8,
n_embd=448`) for ~19.5M parameters and noticeably better samples.


In [ ]:

# ---- model / training hyperparameters ----
block_size = 256     # max context length (sequence length seen during training)
n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.1

# Bigger config (uncomment for ~19.5M params, recommended only with a GPU):
# block_size = 256
# n_layer = 8
# n_head = 8
# n_embd = 448
# dropout = 0.1

model = TinyGPT(
    vocab_size=vocab_size,
    block_size=block_size,
    n_layer=n_layer,
    n_head=n_head,
    n_embd=n_embd,
    dropout=dropout,
).to(device)

n_params = model.num_params()
print(f"Model parameters: {n_params:,} ({n_params/1e6:.2f}M)")



## 5. Correctness check: does the KV-cache actually match the uncached forward pass?

**This is the single most important cell in the notebook.** It's very easy to write a
KV-cache that *runs* without error but produces subtly wrong outputs (e.g. forgetting to
mask during a multi-token prefill, or off-by-one positional embeddings). The only way to
trust a cache implementation is to check it numerically against the ground truth: a normal
full-sequence forward pass.

The test:
1. Run the model on a full random sequence with **no cache** → this is "ground truth" logits.
2. Run the **same** sequence through the cached path: prefill the first few tokens in one
   call (populating the cache), then feed the rest **one token at a time**, each time
   reusing and extending the cache.
3. Concatenate the cached-path logits and compare to the ground truth. They should match
   to floating-point precision.

If this check fails, nothing downstream (correct generation, correct speedup) can be
trusted — so we run it before doing anything else with the cache.


In [ ]:

@torch.no_grad()
def check_kv_cache_correctness(model, vocab_size, seq_len=20, prefill_len=7, device="cpu"):
    model.eval()
    idx = torch.randint(0, vocab_size, (2, seq_len), device=device)  # batch of 2, for good measure

    # 1) ground truth: full forward pass, no cache
    logits_full, _ = model(idx)

    # 2) cached path: prefill first `prefill_len` tokens, then step one at a time
    kv_caches = model.new_kv_cache()
    logits_prefill, _ = model(idx[:, :prefill_len], kv_caches=kv_caches, start_pos=0)

    collected = [logits_prefill]
    cur_len = prefill_len
    for t in range(prefill_len, seq_len):
        step_logits, _ = model(idx[:, t : t + 1], kv_caches=kv_caches, start_pos=cur_len)
        collected.append(step_logits)
        cur_len += 1

    logits_cached = torch.cat(collected, dim=1)

    max_diff = (logits_full - logits_cached).abs().max().item()
    print(f"max |full - cached| logit difference: {max_diff:.3e}")
    assert max_diff < 1e-3, "KV-cache output diverges from the uncached forward pass!"
    print("PASS: KV-cache output matches the full forward pass.")

check_kv_cache_correctness(model, vocab_size, seq_len=24, prefill_len=9, device=device)



## 6. Training loop

Standard recipe:
- **AdamW** optimizer
- **Cosine learning-rate schedule with linear warmup** — warmup avoids early instability
  from large gradients on a freshly-initialized model; cosine decay lets the model "settle"
  as training progresses.
- **Gradient clipping** (clip global norm to 1.0) for additional stability.
- Periodic evaluation on a held-out validation split, averaged over several batches to
  reduce noise.

Adjust `max_steps` based on your hardware: a few hundred steps will show the loss dropping
and start producing recognizable English-ish structure; a few thousand on a GPU gets
noticeably better Shakespeare-flavored text.


In [ ]:

# ---- training hyperparameters ----
max_steps = 2000          # increase for better results if you have a GPU (e.g. 5000-20000)
eval_interval = 200
eval_iters = 50
batch_size = 64
learning_rate = 3e-4
warmup_steps = 100
weight_decay = 0.1
grad_clip = 1.0

optimizer = torch.optim.AdamW(
    model.parameters(), lr=learning_rate, weight_decay=weight_decay, betas=(0.9, 0.95)
)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    # cosine decay from 1.0 -> 0.1 over the remaining steps
    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    progress = min(progress, 1.0)
    return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


In [ ]:

@torch.no_grad()
def estimate_loss(model, block_size, batch_size, device, eval_iters=50):
    model.eval()
    out = {}
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for i in range(eval_iters):
            x, y = get_batch(split, block_size, batch_size, device)
            _, loss = model(x, targets=y)
            losses[i] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


In [ ]:

history = {"step": [], "train_loss": [], "val_loss": []}

model.train()
t0 = time.time()
for step in range(max_steps + 1):
    if step % eval_interval == 0 or step == max_steps:
        losses = estimate_loss(model, block_size, batch_size, device, eval_iters=eval_iters)
        elapsed = time.time() - t0
        print(
            f"step {step:5d} | train loss {losses['train']:.4f} | "
            f"val loss {losses['val']:.4f} | lr {scheduler.get_last_lr()[0]:.2e} | "
            f"elapsed {elapsed:.1f}s"
        )
        history["step"].append(step)
        history["train_loss"].append(losses["train"])
        history["val_loss"].append(losses["val"])

    if step == max_steps:
        break

    xb, yb = get_batch("train", block_size, batch_size, device)
    logits, loss = model(xb, targets=yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()
    scheduler.step()

print(f"\nTotal training time: {time.time() - t0:.1f}s")



### Plot the loss curve

A healthy run shows both train and val loss decreasing together. If val loss starts rising
while train loss keeps falling, that's overfitting — for this dataset size and model size
it's unlikely to be a big problem within a few thousand steps, but it's good practice to
check.


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history["step"], history["train_loss"], label="train loss")
plt.plot(history["step"], history["val_loss"], label="val loss")
plt.xlabel("step")
plt.ylabel("cross-entropy loss")
plt.title("Training curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()



## 7. Generation: greedy/sampled decoding, with and without the KV-cache

Now the payoff. `generate()` below supports two modes:

- **`use_cache=True`** (the interesting path): we "prefill" the cache with the prompt in a
  single forward call, then generate new tokens one at a time. At each step we only run the
  forward pass on the **single new token**, attending against the cache's stored keys/values
  for every previous position — we never recompute K/V for tokens we've already processed.
- **`use_cache=False`** (the naive baseline): at every step, re-run the model on the **entire
  sequence so far** (truncated to the last `block_size` tokens) and only look at the last
  position's logits. This recomputes every key and value from scratch, every step — wasteful,
  but it's the "obvious" implementation if you didn't think about caching.

Both paths use the same sampling logic: divide logits by `temperature` (lower = more
deterministic / greedy, higher = more random), optionally restrict to the `top_k` most
likely next characters, then sample from the resulting distribution.


In [ ]:

@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, top_k=None, use_cache=True):
    """Autoregressive sampling.

    idx: (B, T0) starting token ids (the "prompt").
    Returns: (B, T0 + max_new_tokens) token ids.
    """
    model.eval()

    if use_cache:
        kv_caches = model.new_kv_cache()
        # Prefill: process the whole prompt in ONE forward call, populating the
        # cache with K/V for every prompt position. We only need the logits at
        # the LAST prompt position to predict the first new token.
        prompt = idx[:, -model.block_size :]
        logits, _ = model(prompt, kv_caches=kv_caches, start_pos=0)
        cur_len = prompt.size(1)
        next_logits = logits[:, -1, :]
    else:
        next_logits = None

    for _ in range(max_new_tokens):
        if use_cache:
            logits_step = next_logits
        else:
            # Naive baseline: recompute everything from scratch every step.
            idx_cond = idx[:, -model.block_size :]
            logits_full, _ = model(idx_cond)
            logits_step = logits_full[:, -1, :]

        logits_step = logits_step / temperature
        if top_k is not None:
            v, _ = torch.topk(logits_step, top_k)
            logits_step[logits_step < v[:, [-1]]] = float("-inf")

        probs = F.softmax(logits_step, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)  # (B, 1)
        idx = torch.cat([idx, next_id], dim=1)

        if use_cache:
            if cur_len >= model.block_size:
                # Cache is full (hit the context limit). A production
                # implementation would slide the window and evict old
                # entries; here we simply stop extending for clarity.
                break
            logits, _ = model(next_id, kv_caches=kv_caches, start_pos=cur_len)
            cur_len += 1
            next_logits = logits[:, -1, :]

    return idx


In [ ]:

# Generate a sample from the trained model
torch.manual_seed(42)
prompt = "ROMEO:"
prompt_ids = torch.tensor([encode(prompt)], dtype=torch.long, device=device)

out_ids = generate(model, prompt_ids, max_new_tokens=400, temperature=0.8, top_k=40, use_cache=True)
print(decode(out_ids[0].tolist()))



## 8. Benchmark: cached vs. uncached generation speed

Without a cache, generating token *t* requires recomputing attention over all `t` previous
tokens — and we do this again from scratch for token `t+1`, `t+2`, etc. Total work to
generate `N` tokens scales like `O(N^2)`. With a cache, each new token only costs `O(t)`
work for *that step alone* (attending against the cache), and we never redo previously
finished work — generating `N` tokens costs `O(N^2)` work in attention FLOPs too in
principle (since the cache vectors are still all attended over), but we eliminate ALL the
**redundant K/V projection and earlier-position recomputation**, which dominates wall-clock
time in practice — especially because the no-cache path also reruns every transformer layer
on every earlier token, every single step.

The benchmark below should show a clear, growing speedup as `max_new_tokens` increases.


In [ ]:

def benchmark_generation(model, prompt_ids, max_new_tokens, device):
    # warm up (esp. important on GPU, to avoid measuring kernel/context startup cost)
    _ = generate(model, prompt_ids, max_new_tokens=5, use_cache=True)
    _ = generate(model, prompt_ids, max_new_tokens=5, use_cache=False)
    if device == "cuda":
        torch.cuda.synchronize()

    t0 = time.time()
    _ = generate(model, prompt_ids, max_new_tokens=max_new_tokens, use_cache=True)
    if device == "cuda":
        torch.cuda.synchronize()
    t_cached = time.time() - t0

    t0 = time.time()
    _ = generate(model, prompt_ids, max_new_tokens=max_new_tokens, use_cache=False)
    if device == "cuda":
        torch.cuda.synchronize()
    t_uncached = time.time() - t0

    return t_cached, t_uncached

prompt_ids = torch.tensor([encode("ROMEO:")], dtype=torch.long, device=device)

results = []
for n_new in (50, 100, 200):
    t_cached, t_uncached = benchmark_generation(model, prompt_ids, n_new, device)
    speedup = t_uncached / t_cached if t_cached > 0 else float("inf")
    results.append((n_new, t_cached, t_uncached, speedup))
    print(f"tokens={n_new:4d}  cached={t_cached:6.3f}s  uncached={t_uncached:6.3f}s  speedup={speedup:5.2f}x")


In [ ]:

ns = [r[0] for r in results]
cached_times = [r[1] for r in results]
uncached_times = [r[2] for r in results]

plt.figure(figsize=(8, 5))
plt.plot(ns, cached_times, marker="o", label="with KV-cache")
plt.plot(ns, uncached_times, marker="o", label="without KV-cache (recompute everything)")
plt.xlabel("tokens generated")
plt.ylabel("wall-clock time (s)")
plt.title("Generation time: cached vs. uncached")
plt.legend()
plt.grid(alpha=0.3)
plt.show()



## 9. Recap: what the KV-cache actually does

In a decoder-only transformer, at generation step `t` the model needs, for every layer,
the **key** and **value** vectors of *every* position `0..t` to compute attention for the
new query at position `t`. Crucially, **the key/value vectors for positions `0..t-1` never
change** once computed — they only depend on the (fixed) hidden states at those positions,
not on anything generated afterward. So instead of recomputing them from scratch at every
step, we:

1. Compute K, V for each new token exactly once, the first time it's processed.
2. Store them in a cache (here: simple Python dicts holding tensors, one per layer).
3. At every subsequent generation step, only compute Q, K, V for the **single new token**,
   concatenate the new K, V onto the cached ones, and attend the new query against the
   full (cached + new) set.

This is exactly what production inference engines (vLLM, llama.cpp, HF `transformers`
`generate(use_cache=True)`, etc.) do under the hood, just with extra engineering for memory
management (paging, eviction, batching across requests). The core idea — and the part that's
most error-prone to implement by hand — is exactly what we verified numerically in section 5:
**the cached computation must be mathematically identical to the uncached one**, just
organized to avoid redundant work.

### Ideas to extend this notebook
- **Sliding-window cache**: when the cache hits `block_size`, evict the oldest entries
  instead of stopping (lets you generate text longer than the training context).
- **Multi-query / grouped-query attention**: share K/V across multiple query heads to shrink
  the cache's memory footprint — the main lever real inference systems pull to serve longer
  contexts and bigger batches.
- **Batched generation with different prompt lengths**: requires padding + attention masks
  that interact with the cache logic above.
- **Larger model / longer training**: bump `n_layer`/`n_embd` toward the ~30M end of the
  range and train for more steps on a GPU for noticeably more coherent samples.
